In [ ]:
# Machine-specific paths live in config.yaml next to this notebook (gitignored).
# Copy config.example.yaml -> config.yaml and fill in your paths.
from phoreal.config import load_config
cfg = load_config('config.yaml')

### Import packages

In [ ]:
%load_ext autoreload
%autoreload 2

#Import general libs you'll need
import os
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import numpy as np

# Import functions from main
from phoreal.agg_configs import ATL_AGG_CONFIG

# Import functions from the utils
from phoreal.io import readers, processing
from phoreal import analysis
from phoreal import plotting as plotter
from phoreal.als.swath import create_als_swath, transform_als_swath
from phoreal.geo.datum_transforms import convert_3d_nad83_to_wgs84
from phoreal.geo.datum_transforms import get_geoid_height

import matplotlib.pyplot as plt


### Identify files



In [ ]:
f_dir = cfg['esf_hits_dir']

# f_list = os.listdir(f_dir)
f_list = [f for f in os.listdir(f_dir) if f.endswith('_target_ph.pqt')]

df_list = []
total = len(f_list)
rejected = 0
for f in f_list:
    df = pd.read_parquet(os.path.join(f_dir,f))
    df['file_name'] = f
    df = df[df['signal_conf_ph0'] > 1]
    # median_roof = np.median(df.h_ph[(df.target_class == 'roof') & (df.dist_to_wall) > 0])
    # median_ground = np.median(df.h_ph[(df.target_class == 'ground') & (df.dist_to_wall) < 0])



    if (len(df[df.target_class == 'ground']) > 2) &\
        (len(df[df.target_class == 'roof']) > 2) &\
        (len(df[df.atl08_class == 1]) > 2) &\
        (np.max(df.dist_to_wall[df.target_class == 'ground']) < 7)  &\
        (np.min(df.dist_to_wall[df.target_class == 'roof']) > -7):

        df_list.append(df)
    else:
        # print(f'Rejected {f}')
        rejected = rejected + 1

valid_df = pd.concat(df_list,ignore_index=True)

# valid_df['target_class'][(valid_df.target_class == 'noise') & (valid_df.dist_to_wall > 0)] = 'roof'

# valid_df['target_class'][(valid_df.target_class == 'roof') & (valid_df.dist_to_wall > 0)] = 'roof'


print(f'{rejected}/{total}')
print(f'{total - rejected} usable building strikes')





In [ ]:
num=22
print(f_list[num])
df = pd.read_parquet(os.path.join(f_dir,f_list[num]))
plt.plot(df.dist_to_wall,df.h_ph,'.')


In [ ]:
len(np.unique(valid_df.file_name))


In [ ]:
from building_processing import compute_esf


esf_df = compute_esf(valid_df, min_dist=-15.0, max_dist=15.0, bin_size=1)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(esf_df.distance_to_wall,esf_df.esf_ratio)

In [ ]:
unique_files = np.unique(valid_df.file_name)


test_df = valid_df[valid_df.file_name == unique_files[25]]
print(np.unique(test_df.file_name))

plt.title(np.unique(test_df.file_name)[0])
plt.xlabel('Distance to Wall (m)')
plt.ylabel('Height (m)')
plt.plot(test_df.dist_to_wall[test_df.target_class=='noise'],test_df.h_norm[test_df.target_class=='noise'],'.k',label='noise')
plt.plot(test_df.dist_to_wall[test_df.target_class=='ground'],test_df.h_norm[test_df.target_class=='ground'],'.r',label='ground')
plt.plot(test_df.dist_to_wall[test_df.target_class=='roof'],test_df.h_norm[test_df.target_class=='roof'],'.g',label='roof')
plt.legend()

In [ ]:
len(unique_files)

In [ ]:
unique_files[13]

In [ ]:
unique_files[0]

In [ ]:
unique_files[11]

In [ ]:
from building_processing import extract_laser_footprint

x_smooth, y_esf_fit, y_pdf, footprint_diameter = extract_laser_footprint(esf_df)


In [ ]:
plt.plot(esf_df.distance_to_wall,esf_df.esf_ratio, label='Measured ESF')
plt.plot(x_smooth,y_esf_fit, label='Ideal ESF Fit')
plt.title('ICESat-2 Edge Spread Function: Empirical vs. Ideal Gaussian Fit')
plt.ylabel('Protion of Roof Returns')
plt.xlabel('Dist to Wall (m)')


In [ ]:
# plt.plot(x_smooth,y_pdf)
plt.plot(x_smooth, y_pdf)


In [ ]:
from building_processing import extract_laser_footprint2

curves, metrics = extract_laser_footprint2(esf_df)
print(metrics)


In [ ]:
plt.plot(esf_df.distance_to_wall,esf_df.esf_ratio, label='Measured ESF')
plt.plot(curves['x_empirical'], curves['y_esf_empirical'],label='Smoothed ESF')
plt.plot(curves['x_ideal'], curves['y_esf_ideal'],'g--',label='Ideal Fit')
plt.legend()
plt.title('ICESat-2 Edge Spread Function: Empirical vs. Ideal Gaussian Fit')
plt.ylabel('Protion of Roof Returns')
plt.xlabel('Dist to Wall (m)')

In [ ]:
plt.plot(curves['x_empirical'], curves['y_pdf_empirical'],label='Empircal Fit')
plt.plot(curves['x_ideal'], curves['y_pdf_ideal'],'g--',label='Ideal fit')
plt.xlabel('Dist to Wall (m)')
plt.ylabel('Probability Density (1/m)')
plt.legend()
plt.title('Empirical vs. Ideal PDF Best-Fit')

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

class RapidStrikeFilter:
    def __init__(self, df_list):
        self.df_list = df_list
        self.index = 0
        self.decisions = [] # List of indices to keep
        
        # UI Elements
        self.progress = widgets.IntProgress(value=0, max=len(df_list), description='Progress:')
        self.label = widgets.Label(value=f"Strike 0 of {len(df_list)}")
        
        self.btn_keep = widgets.Button(description="Keep (Good)", button_style='success', icon='check')
        self.btn_drop = widgets.Button(description="Drop (Bad)", button_style='danger', icon='trash')
        self.btn_undo = widgets.Button(description="Undo", button_style='warning', icon='undo')
        
        self.out = widgets.Output()
        
        # Event Handlers
        self.btn_keep.on_click(self.keep)
        self.btn_drop.on_click(self.drop)
        self.btn_undo.on_click(self.undo)
        
        # Layout
        controls = widgets.HBox([self.btn_keep, self.btn_drop, self.btn_undo])
        display(widgets.VBox([self.progress, self.label, controls, self.out]))
        
        self.show_current()

    def show_current(self):
            with self.out:
                clear_output(wait=True)
                if self.index >= len(self.df_list):
                    print(f"🎉 Done! You kept {len(self.decisions)} strikes out of {len(self.df_list)}.")
                    return

                df = self.df_list[self.index]
                name = np.unique(df.file_name)[0]
                self.label.value = f"Processing Strike {self.index} of {len(self.df_list)}"
                
                # IMPORTANT: Removed sharex=True because the scales are totally different
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
                
                colors = {'ground': '#8B4513', 'roof': '#1E90FF', 'building': '#1E90FF', 'noise': '#A9A9A9'}
                
                for cls, group in df.groupby('target_class'):
                    c = colors.get(cls, 'black')
                    # Plot 1: Relative distance (Physics View)
                    ax1.scatter(group['dist_to_wall'], group['h_norm'], label=cls, s=15, alpha=0.6, c=c)
                    # Plot 2: Absolute Along-track (Geographic View)
                    ax2.scatter(group['alongtrack'], group['h_ph'], label=cls, s=15, alpha=0.6, c=c)

                # --- Plot 1 Settings: Relative View ---
                ax1.axvline(0, color='red', linestyle='--', alpha=0.8, label='Building Edge')
                ax1.axhline(0, color='black', alpha=0.3, linestyle=':')
                ax1.set_title("Relative: dist_to_wall vs h_norm")
                ax1.set_ylabel("Height Above Ground [m]")
                ax1.set_xlabel("Dist to Wall [m]")
                ax1.set_ylim(-5, 35) # Fixed zoom for consistency
                ax1.grid(True, alpha=0.2)

                # --- Plot 2 Settings: Absolute View ---
                # To find the wall in alongtrack space, we find where dist_to_wall is closest to 0
                # Sort by absolute distance and grab the alongtrack value of the top result
                idx_at_wall = df['dist_to_wall'].abs().idxmin()
                wall_at_coord = df.loc[idx_at_wall, 'alongtrack']
                
                ax2.axvline(wall_at_coord, color='red', linestyle='--', alpha=0.8, label='Approx Edge')
                ax2.set_title("Absolute: alongtrack vs h_ph")
                ax2.set_ylabel("Elevation [m]")
                ax2.set_xlabel("Along-track Distance [m]")
                
                # Robust dynamic zoom for h_ph
                signal_df = df[df['target_class'].isin(['roof', 'ground', 'building'])]
                if not signal_df.empty:
                    ymin, ymax = np.percentile(signal_df['h_ph'], [2, 98])
                    ax2.set_ylim(ymin - 10, ymax + 10)
                
                ax2.grid(True, alpha=0.2)
                # Put the legend between the plots or off to the side
                ax2.legend(loc='upper left', bbox_to_anchor=(1.02, 1))

                plt.suptitle(f"Strike Index: {self.index} | Reference ID: {name}")
                plt.tight_layout()
                plt.show()

    def keep(self, _):
        self.decisions.append(self.index)
        self.next()

    def drop(self, _):
        # We just don't add to the index list
        self.next()

    def undo(self, _):
        if self.index > 0:
            self.index -= 1
            if self.index in self.decisions:
                self.decisions.remove(self.index)
            self.progress.value = self.index
            self.show_current()

    def next(self):
        self.index += 1
        self.progress.value = self.index
        self.show_current()

    def get_clean_data(self):
        """Returns the list of DataFrames that passed the Hot Dog test."""
        return [self.df_list[i] for i in self.decisions]

In [ ]:
f_dir = cfg['esf_hits_dir_v2']

# f_list = os.listdir(f_dir)
f_list = [f for f in os.listdir(f_dir) if f.endswith('.pqt')]

df_list_process = []


In [ ]:
total = len(f_list)
rejected = 0
kept = 0
for f in f_list:
    df = pd.read_parquet(os.path.join(f_dir,f))
    df['file_name'] = f
    
    if (len(df[df.target_class == 'ground']) > 1) &\
        (len(df[df.target_class == 'roof']) > 1) &\
        (len(df[df.atl08_class == 1]) > 2):
        if (np.median(df.h_norm[df.target_class == 'roof']) > 7.5) &\
            (np.std(df.h_ph[df.target_class == 'ground']) < 2)  &\
            (np.median(df.dist_to_wall[df.target_class == 'roof']) > 0) &\
            (np.median(df.dist_to_wall[df.target_class == 'ground']) < 0):
            df_list_process.append(df)
            kept = kept + 1
    else:
        # print(f'Rejected {f}')
        rejected = rejected + 1

valid_df_exp = pd.concat(df_list_process,ignore_index=True)


print(f'{kept}/{total}')
print(f'{total - kept} rejected building strikes')

In [ ]:
filter_tool = RapidStrikeFilter(df_list)

In [ ]:
len(filter_tool.decisions)
for i in filter_tool.decisions:
    print(i)

In [ ]:
decision_list = [df_list[i] for i in filter_tool.decisions]


In [ ]:
decision_list

In [ ]:

decision_df = pd.concat(decision_list,ignore_index=True)
esf_df = compute_esf(decision_df, min_dist=-15.0, max_dist=15.0, bin_size=0.5)

In [ ]:
%load_ext autoreload
%autoreload 2
from building_processing import extract_laser_footprint2

curves, metrics = extract_laser_footprint2(esf_df)
print(f'Footprint Diameter: {metrics["diam_86_5"]}')
print(f'FWHM: {metrics["fwhm"]}')
print(metrics)

In [ ]:
plt.plot(esf_df.distance_to_wall,esf_df.esf_ratio, label='Measured ESF')
plt.plot(curves['x_empirical'], curves['y_esf_empirical'],label='Smoothed ESF')
plt.plot(curves['x_ideal'], curves['y_esf_ideal'],'g--',label='Ideal ESF')
plt.legend()
plt.title('ICESat-2 Edge Spread Function: Empirical vs. Ideal Gaussian Fit')
plt.ylabel('Protion of Roof Returns')
plt.xlabel('Dist to Wall (m)')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 6))

# 1. Measured ESF: Small dots, semi-transparent, gray to show the observation density
plt.plot(esf_df.distance_to_wall, esf_df.esf_ratio, 
         marker='.', linestyle='none', color='gray', alpha=0.5, 
         markersize=10, label='Measured ESF (Raw Photons)')

# 2. Smoothed ESF: The "Hero" curve. Thick, solid, and vibrant.
plt.plot(curves['x_empirical'], curves['y_esf_empirical'], 
         color='#1f77b4', linewidth=3.5, label='Empirical ESF (Smoothed)')

# 3. Ideal ESF: High-contrast dashed line for comparison
plt.plot(curves['x_ideal'], curves['y_esf_ideal'], 
         color='#d62728', linestyle='--', linewidth=2.5, zorder=5, 
         label='Theoretical Gaussian Ideal')

# Formatting for ForestSat Standards
plt.axvline(0, color='black', linestyle='-', alpha=0.3, linewidth=1) # The Wall
plt.axhline(0.5, color='black', linestyle=':', alpha=0.3) # The 50% power point

plt.title('ICESat-2 Edge Step Function (Strong beams/Night acquistions)', 
          fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Proportion of Roof Returns', fontsize=12)
plt.xlabel('Distance to Wall (m)', fontsize=12)

plt.grid(True, linestyle=':', alpha=0.5)
plt.legend(loc='upper left', framealpha=0.9)

# Focusing on the transition zone
plt.xlim(-12, 12) 
plt.ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
plt.plot(curves['x_empirical'], curves['y_pdf_empirical'],label='Measured PDF')
plt.plot(curves['x_empirical'], curves['y_pdf_smoothed'],label='Smoothed PDF')
plt.plot(curves['x_ideal'], curves['y_pdf_ideal'],'g--',label='Ideal PDF')
plt.xlabel('Dist to Wall (m)')
plt.ylabel('Probability Density (1/m)')
plt.legend()
plt.title('Strong-Beam/Night-time Power Density Function')

In [ ]:
plt.figure(figsize=(8, 5))

# 1. Measured (Raw): Push to background, use dots or thin semi-transparent line
plt.plot(curves['x_empirical'], curves['y_pdf_empirical'], 
         marker='.', linestyle='none', color='gray', alpha=0.4, 
         markersize=10, label='Measured (Raw Data)')

# 2. Smoothed: Solid, bold, visually dominant
plt.plot(curves['x_empirical'], curves['y_pdf_smoothed'], 
         color='#1f77b4', linewidth=3, label='Empirical Fit (Smoothed)')

# 3. Ideal: Contrasting color, dashed, brought to the absolute front
plt.plot(curves['x_ideal'], curves['y_pdf_ideal'], 
         color='#d62728', linestyle='--', linewidth=2.5, zorder=5, 
         label='Theoretical Gaussian Ideal')

# Formatting
plt.xlabel('Distance to Wall (m)', fontsize=12)
plt.ylabel('Probability Density (1/m)', fontsize=12)
plt.title('ICESat-2 Power Density Function (Strong beam/night acquistions)', fontsize=14, fontweight='bold')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(framealpha=0.9, loc='upper right')
plt.xlim(-10, 10) # Zoom in to cut out the dead space on the tails
plt.tight_layout()
plt.show()

In [ ]:
# Set up a grid with a tall top plot and short bottom plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7), 
                               gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

# --- Top Panel: The PDFs ---
ax1.plot(curves['x_empirical'], curves['y_pdf_empirical'], '.', color='gray', alpha=0.3, label='Measured')
ax1.plot(curves['x_empirical'], curves['y_pdf_smoothed'], color='#1f77b4', linewidth=2.5, label='Empirical Fit')
ax1.plot(curves['x_ideal'], curves['y_pdf_ideal'], color='#d62728', linestyle='--', linewidth=2, zorder=5, label='Theoretical Ideal')

ax1.set_ylabel('Probability Density (1/m)', fontsize=12)
ax1.set_title('ICESat-2 Footprint: Empirical vs. Ideal PDF', fontsize=14, fontweight='bold')
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(loc='upper right')

# --- Bottom Panel: The Residuals ---
# Note: This assumes x_empirical and x_ideal are aligned. 
# If they are different lengths, you may need to interpolate one to match the other first!
residuals = curves['y_pdf_empirical'] - curves['y_pdf_smoothed']

ax2.fill_between(curves['x_empirical'], 0, residuals, color='purple', alpha=0.3)
ax2.plot(curves['x_empirical'], residuals, color='purple', linewidth=2)
ax2.axhline(0, color='black', linestyle='--', alpha=0.8)

ax2.set_ylabel('$\Delta$ Density', fontsize=12)
ax2.set_xlabel('Distance to Wall (m)', fontsize=12)
ax2.grid(True, linestyle=':', alpha=0.6)

# Clean up layout
plt.xlim(-10, 10)
plt.tight_layout()
plt.show()

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np

class RapidStrikeFilter:
    def __init__(self, data_pairs):
        """
        data_pairs: A list of tuples containing (df_is2, df_als) for each strike.
        """
        self.data_pairs = data_pairs
        self.index = 0
        self.decisions = {} # Dictionary mapping index -> 'keep', 'drop', or 'edit'
        
        # UI Elements
        self.progress = widgets.IntProgress(value=0, max=len(data_pairs), description='Progress:')
        self.label = widgets.Label(value=f"Strike 0 of {len(data_pairs)}")
        
        self.btn_keep = widgets.Button(description="Keep (Good)", button_style='success', icon='check')
        self.btn_edit = widgets.Button(description="Needs Edits", button_style='info', icon='pencil')
        self.btn_drop = widgets.Button(description="Drop (Bad)", button_style='danger', icon='trash')
        self.btn_undo = widgets.Button(description="Undo", button_style='warning', icon='undo')
        
        self.out = widgets.Output()
        
        # Event Handlers
        self.btn_keep.on_click(self.keep)
        self.btn_edit.on_click(self.edit)
        self.btn_drop.on_click(self.drop)
        self.btn_undo.on_click(self.undo)
        
        # Layout
        controls = widgets.HBox([self.btn_keep, self.btn_edit, self.btn_drop, self.btn_undo])
        display(widgets.VBox([self.progress, self.label, controls, self.out]))
        
        self.show_current()

    def show_current(self):
        with self.out:
            clear_output(wait=True)
            if self.index >= len(self.data_pairs):
                kept = sum(1 for v in self.decisions.values() if v == 'keep')
                edited = sum(1 for v in self.decisions.values() if v == 'edit')
                print(f"🎉 Done! Kept: {kept} | Needs Edits: {edited} | Dropped: {len(self.data_pairs) - kept - edited}")
                return

            df_ph, df_als = self.data_pairs[self.index]
            
            # Safely grab the file name if it exists
            name = np.unique(df_ph['file_name'])[0] if 'file_name' in df_ph.columns else "Unknown"
            self.label.value = f"Processing Strike {self.index} of {len(self.data_pairs)}"
            
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
            
            colors = {'ground': '#8B4513', 'roof': '#1E90FF', 'building': '#1E90FF', 'noise': '#A9A9A9'}
            
            # --- Plot 1 Settings: Relative View (h_norm) ---
            for cls, group in df_ph.groupby('target_class'):
                c = colors.get(cls, 'black')
                ax1.scatter(group['dist_to_wall'], group['h_norm'], label=cls, s=15, alpha=0.6, c=c)

            ax1.axvline(0, color='red', linestyle='--', alpha=0.8, label='Building Edge')
            ax1.axhline(0, color='black', alpha=0.3, linestyle=':')
            ax1.set_title("Relative: dist_to_wall vs h_norm")
            ax1.set_ylabel("Height Above Ground [m]")
            ax1.set_xlabel("Dist to Wall [m]")
            ax1.set_ylim(-5, 35) # Fixed zoom for consistency
            ax1.grid(True, alpha=0.2)

            # --- Plot 2 Settings: Absolute View with ALS Context ---
            
            # 1. Plot the dense ALS data in the background
            if df_als is not None and not df_als.empty:
                ax2.scatter(df_als['dist_to_wall'], df_als['ellip_h'], 
                            color=[0.8,0.8,0.8], s=2, alpha=0.2, label='ALS Surface', zorder=1)

            # 2. Plot the IS2 Photons over the ALS
            for cls, group in df_ph.groupby('target_class'):
                c = colors.get(cls, 'black')
                ax2.scatter(group['dist_to_wall'], group['h_ph'], label=cls, s=15, alpha=0.8, c=c, zorder=5)

            ax2.axvline(0, color='red', linestyle='--', alpha=0.8, label='Building Edge')
            ax2.set_title("Absolute Context: dist_to_wall vs Elevation")
            ax2.set_ylabel("Elevation (ellip_h / h_ph) [m]")
            ax2.set_xlabel("Dist to Wall [m]")
            
            # Robust dynamic zoom for Absolute Elevation
            signal_df = df_ph[df_ph['target_class'].isin(['roof', 'ground', 'building'])]
            if not signal_df.empty:
                ymin, ymax = np.percentile(signal_df['h_ph'], [2, 98])
                ax2.set_ylim(ymin - 5, ymax + 10)
            elif df_als is not None and not df_als.empty:
                # Fallback to ALS bounds if IS2 didn't classify anything
                ymin, ymax = np.percentile(df_als['ellip_h'], [2, 98])
                ax2.set_ylim(ymin - 5, ymax + 10)
            
            ax2.grid(True, alpha=0.2)
            
            # Handle legend cleanly
            handles, labels = ax2.get_legend_handles_labels()
            by_label = dict(zip(labels, handles)) # Removes duplicates
            ax2.legend(by_label.values(), by_label.keys(), loc='upper left', bbox_to_anchor=(1.02, 1))

            plt.suptitle(f"Strike Index: {self.index} | Reference ID: {name}")
            plt.tight_layout()
            plt.show()

    def keep(self, _):
        self.decisions[self.index] = 'keep'
        self.next()

    def edit(self, _):
        self.decisions[self.index] = 'edit'
        self.next()

    def drop(self, _):
        self.decisions[self.index] = 'drop'
        self.next()

    def undo(self, _):
        if self.index > 0:
            self.index -= 1
            # Remove the decision from the dictionary
            self.decisions.pop(self.index, None)
            self.progress.value = self.index
            self.show_current()

    def next(self):
        self.index += 1
        self.progress.value = self.index
        self.show_current()

    def get_results(self):
        """Returns a dictionary separating the kept, edit, and dropped pairs."""
        results = {'keep': [], 'edit': [], 'drop': []}
        for i, pair in enumerate(self.data_pairs):
            decision = self.decisions.get(i, 'drop') # Default to drop if skipped somehow
            results[decision].append(pair)
        return results